# imports

In [1]:
import os
import random
import numpy as np
import torch
import torchaudio
import librosa
from pathlib import Path
from datasets import load_from_disk, Dataset, DatasetDict, Audio
from typing import Optional
from tqdm.auto import tqdm


# Load Data

In [2]:
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / '.git').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH       = PROJECT_ROOT / 'data' / 'synthetic' / 'v2'
SYNTHETIC_AUDIO_DIR = PROJECT_ROOT / 'data' / 'synthetic' / 'audio'
TEANGLANN_AUDIO_DIR = PROJECT_ROOT / 'data' / 'teanglann' / 'wav_files'
OUTPUT_PATH        = PROJECT_ROOT / 'data' / 'synthetic' / 'l2_training_set'


In [3]:
paired_ds = load_from_disk(str(DATASET_PATH))
print(paired_ds)

Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Dataset({
    features: ['audio', 'phonetic', 'English ASR transcriptions', 'synthetic_audio_path'],
    num_rows: 19136
})


# Inspect Data

In [4]:
paired_ds[0]

{'audio': {'path': 'carbad.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'kʌɾəbʌd',
 'synthetic_audio_path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/0.mp3'}

Not great that I am saving audio inconsistently, remember not to do this in the future.

# Prepare Data

First let's load the audio into the dataset so the audio is presented in the same way

In [5]:
from datasets import Audio

paired_ds = paired_ds.cast_column("synthetic_audio_path", Audio(sampling_rate=16000))
paired_ds = paired_ds.rename_column("synthetic_audio_path", "synthetic_audio")

In [6]:
paired_ds[0]

{'audio': {'path': 'carbad.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'kʌɾəbʌd',
 'synthetic_audio': {'path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/0.mp3',
  'array': array([1.22781785e-11, 4.09272616e-12, 6.82121026e-12, ...,
         1.23028804e-04, 1.61062751e-04, 3.50766568e-05], shape=(26496,)),
  'sampling_rate': 16000}}

forgot to normalize transcriptions to same wav2vec2-friendly format as the teanglann scrapings. better for spaces between characters to keep diacritics together with their phones

In [7]:
import json


SYNTH_VOCAB = {' ': 0, 'aɪ': 1, 'aʊ': 2, 'b': 3, 'd': 4, 'eɪ': 5, 'f': 6, 'g': 7,
               'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'l̩': 13, 'm': 14, 'm̩': 15,
               'n': 16, 'n̩': 17, 'oʊ': 18, 'p': 19, 's': 20, 't': 21, 'u': 22, 'v': 23,
               'w': 24, 'z': 25, 'æ': 26, 'ð': 27, 'ŋ': 28, 'ŋ̍': 29, 'ɑ': 30, 'ɔ': 31,
               'ɔɪ': 32, 'ə': 33, 'ə̥': 34, 'ɚ': 35, 'ɛ': 36, 'ɝ': 37, 'ɦ': 38, 'ɨ': 39,
               'ɪ': 40, 'ɹ': 41, 'ɾ': 42, 'ɾ̃': 43, 'ʃ': 44, 'ʉ': 45, 'ʊ': 46, 'ʌ': 47,
               'ʒ': 48, 'ʔ': 49, 'ʤ': 50, 'ʧ': 51, 'θ': 52}

# Partition into 2-codepoint and 1-codepoint sets for greedy matching
_PHONES_2 = {k for k in SYNTH_VOCAB if len(k) == 2 and k != ' '}
_PHONES_1 = {k for k in SYNTH_VOCAB if len(k) == 1 and k != ' '}


def _space_separate(ipa: str) -> str:
    """Insert spaces between phones in a collapsed synthetic IPA string."""
    if ' ' in ipa:          # already separated
        return ipa
    phones, i = [], 0
    while i < len(ipa):
        if ipa[i:i+2] in _PHONES_2:
            phones.append(ipa[i:i+2])
            i += 2
        else:
            if ipa[i] in _PHONES_1:
                phones.append(ipa[i])
            i += 1
    return ' '.join(phones)


paired_ds = paired_ds.map(
    lambda item: {'English ASR transcriptions': _space_separate(item['English ASR transcriptions'])},
    desc='Normalising synthetic IPA',
)
print('Before:', 'kʌɾəbʌd')
print('After: ', _space_separate('kʌɾəbʌd'))

Normalising synthetic IPA: 100%|██████████| 19136/19136 [00:02<00:00, 6791.58 examples/s] 

Before: kʌɾəbʌd
After:  k ʌ ɾ ə b ʌ d


In [8]:
paired_ds[0]

{'audio': {'path': 'carbad.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'k ʌ ɾ ə b ʌ d',
 'synthetic_audio': {'path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/0.mp3',
  'array': array([1.22781785e-11, 4.09272616e-12, 6.82121026e-12, ...,
         1.23028804e-04, 1.61062751e-04, 3.50766568e-05], shape=(26496,)),
  'sampling_rate': 16000}}

## strip diacritics

In [9]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.data_handling.collapse_phonemes import collapse_phones

In [10]:
def normalize_row(row):
    row['phonetic'] = "".join(collapse_phones(row['phonetic']))
    row['English ASR transcriptions'] = "".join(collapse_phones(row['English ASR transcriptions']))
    return row

In [11]:
paired_ds = paired_ds.map(
    lambda row: normalize_row(row),
    load_from_cache_file=False,
    desc="Normalizing phonetic columns"
)

Normalizing phonetic columns: 100%|██████████| 19136/19136 [00:03<00:00, 5919.90 examples/s]


In [12]:
paired_ds[0]

{'audio': {'path': 'carbad.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'k ʌ ɾ ə b ʌ d',
 'synthetic_audio': {'path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/0.mp3',
  'array': array([1.22781785e-11, 4.09272616e-12, 6.82121026e-12, ...,
         1.23028804e-04, 1.61062751e-04, 3.50766568e-05], shape=(26496,)),
  'sampling_rate': 16000}}

In [13]:
len(paired_ds[0]['phonetic'])

13

## make paired set

In [21]:
TARGET_SR = 16_000
IPA_SEPARATOR = " "
RANDOM_SEED = 42


def _to_mono_16k(audio_array: np.ndarray, source_sr: int) -> np.ndarray:
    """Resample and convert to mono float32 at TARGET_SR."""
    waveform = torch.from_numpy(audio_array.astype(np.float32))
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if source_sr != TARGET_SR:
        resampler = torchaudio.transforms.Resample(orig_freq=source_sr, new_freq=TARGET_SR)
        waveform = resampler(waveform)
    return waveform.squeeze(0).numpy()


def _make_l2_pair(item: dict, idx: int) -> dict:
    """Map function: concatenate native + TTS audio for one word."""
    ref_audio = _to_mono_16k(item["audio"]["array"], item["audio"]["sampling_rate"])

    synth_array = item["synthetic_audio"]["array"]
    synth_sr = item["synthetic_audio"]["sampling_rate"]
    synth_audio = _to_mono_16k(synth_array, synth_sr)

    # Per-item seeding gives the same reproducibility as a shared rng.
    if random.Random(RANDOM_SEED + idx).random() < 0.5: # Vary order so ordering isn't introducing bias
        combined = np.concatenate([ref_audio, synth_audio])
        ipa = item["phonetic"] + IPA_SEPARATOR + item["English ASR transcriptions"]
        order = "ref_first"
    else:
        combined = np.concatenate([synth_audio, ref_audio])
        ipa = item["English ASR transcriptions"] + IPA_SEPARATOR + item["phonetic"]
        order = "synth_first"

    return {"audio": {"array": combined, "sampling_rate": TARGET_SR}, "ipa": ipa, "order": order}


def build_l2_training_set(dataset: Dataset, seed: Optional[int] = RANDOM_SEED) -> Dataset:
    """
    Build an adversarially-concatenated dataset directly from the paired dataset.

    Input columns:  audio, phonetic, synthetic_audio, English ASR transcriptions
    Output columns: audio, ipa, order
    """
    dataset = dataset.cast_column("audio", Audio(sampling_rate=TARGET_SR))
    out_ds = dataset.map(
        _make_l2_pair,
        with_indices=True,
        remove_columns=["phonetic", "synthetic_audio", "English ASR transcriptions"],
        desc="Building L2 training set",
    )
    return out_ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))


In [17]:
l2_ds = build_l2_training_set(paired_ds)

Building L2 training set: 100%|██████████| 19136/19136 [14:05<00:00, 22.64 examples/s]  


In [22]:
print(l2_ds)
print('\nSample IPA:', l2_ds[0]['ipa'])

Dataset({
    features: ['audio', 'ipa', 'order'],
    num_rows: 19136
})

Sample IPA: k ʌ ɾ ə b ʌ d k a ɾ ə b ə d


# Split dataset
.8/.1/.1 split to be comparable to other models

In [23]:
l2_ds

Dataset({
    features: ['audio', 'ipa', 'order'],
    num_rows: 19136
})

In [24]:
ds_temp = l2_ds.train_test_split(test_size=0.2, seed=RANDOM_SEED)
ds_final = ds_temp["test"].train_test_split(test_size=0.5, seed=RANDOM_SEED)

final_ds = DatasetDict({
    "train":      ds_temp["train"],
    "validation": ds_final["train"],
    "test":       ds_final["test"],
})

In [25]:
final_ds

DatasetDict({
    train: Dataset({
        features: ['audio', 'ipa', 'order'],
        num_rows: 15308
    })
    validation: Dataset({
        features: ['audio', 'ipa', 'order'],
        num_rows: 1914
    })
    test: Dataset({
        features: ['audio', 'ipa', 'order'],
        num_rows: 1914
    })
})

# Save datasets

In [26]:
print(OUTPUT_PATH)

/home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set


In [27]:
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
final_ds.save_to_disk(str(OUTPUT_PATH))
print(f'Saved to {OUTPUT_PATH}')


Saving the dataset (1/1 shards): 100%|██████████| 1914/1914 [00:00<00:00, 4007.83 examples/s]

Saved to /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set


# Build IPA vocab

In [28]:
train_phonetics = [phone for x in final_ds["train"] for phone in x['ipa'].split()]
valid_phonetics = [phone for x in final_ds["validation"] for phone in x['ipa'].split()]
test_phonetics = [phone for x in final_ds["test"] for phone in x['ipa'].split()]

print("num of train phones:\t", len(set(train_phonetics)))
print("num of valid phones:\t", len(set(valid_phonetics)))
print("num of test phones:\t", len(set(test_phonetics)))

num of train phones:	 79
num of valid phones:	 78
num of test phones:	 78


In [29]:
vocab_train = list(set(train_phonetics)) + [' ']
vocab_valid = list(set(valid_phonetics)) + [' ']
vocab_test  = list(set(test_phonetics)) + [' ']

In [30]:
vocab_list = list(set(vocab_train + vocab_valid + vocab_test))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}

print(vocab_dict)

{' ': 0, 'a': 1, 'ai': 2, 'au': 3, 'aɪ': 4, 'aʊ': 5, 'aː': 6, 'b': 7, 'bʲ': 8, 'c': 9, 'd': 10, 'dʲ': 11, 'e': 12, 'eɪ': 13, 'eː': 14, 'f': 15, 'fʲ': 16, 'h': 17, 'hʲ': 18, 'i': 19, 'ia': 20, 'iː': 21, 'iˑə': 22, 'j': 23, 'k': 24, 'l': 25, 'lʲ': 26, 'm': 27, 'mʲ': 28, 'n': 29, 'nʲ': 30, 'o': 31, 'oʊ': 32, 'oː': 33, 'p': 34, 'pʲ': 35, 's': 36, 't': 37, 'tʲ': 38, 'u': 39, 'ua': 40, 'uː': 41, 'uˑə': 42, 'v': 43, 'vʲ': 44, 'w': 45, 'x': 46, 'z': 47, 'zʲ': 48, 'æ': 49, 'ç': 50, 'ð': 51, 'ŋ': 52, 'ɑ': 53, 'ɒ': 54, 'ɔ': 55, 'ɔɪ': 56, 'ə': 57, 'ɚ': 58, 'ɛ': 59, 'ɝ': 60, 'ɟ': 61, 'ɡ': 62, 'ɣ': 63, 'ɦ': 64, 'ɨ': 65, 'ɪ': 66, 'ɲ': 67, 'ɹ': 68, 'ɾ': 69, 'ɾʲ': 70, 'ʃ': 71, 'ʉ': 72, 'ʊ': 73, 'ʌ': 74, 'ʒ': 75, 'ʔ': 76, 'ʤ': 77, 'ʧ': 78, 'θ': 79}


In [31]:
# make the space more intuitive to understand
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

82

In [ ]:
# save vocab.json
import json
with open(str(OUTPUT_PATH)+"/vocab.json", 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

# Upload to Kaggle

In [4]:
import kagglehub
kagglehub.login()

In [ ]:
# For example, to upload a new dataset (or version) at:
# - https://www.kaggle.com/datasets/bricevergnou/spotify-recommendation
# 
# You would use the following handle: `bricevergnou/spotify-recommendation`
handle = 'petercady/L2Synth_teanglann_paired'
local_dataset_dir = str(OUTPUT_PATH)

# Create a new dataset
#kagglehub.dataset_upload(handle, local_dataset_dir)

# You can then create a new version of this existing dataset and include version notes (optional).
kagglehub.dataset_upload(handle, local_dataset_dir, version_notes='improved synthetic audio')

# You can also specify a list of patterns for files/dirs to ignore.
# These patterns are combined with `kagglehub.datasets.DEFAULT_IGNORE_PATTERNS` 
# to determine which files and directories to exclude. 
# To ignore entire directories, include a trailing slash (/) in the pattern.
#kagglehub.dataset_upload(handle, local_dataset_dir, ignore_patterns=["original/", "*.tmp"])

Uploading Dataset https://kaggle.com/datasets/petercady/L2Synth_teanglann_paired ...
Starting upload for file /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set/vocab.json


Uploading: 100%|██████████| 1.05k/1.05k [00:00<00:00, 1.96kB/s]

Upload successful: /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set/vocab.json (1KB)
Starting upload for file /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set/dataset_dict.json



Uploading: 100%|██████████| 43.0/43.0 [00:00<00:00, 94.2B/s]

Upload successful: /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set/dataset_dict.json (43B)
Starting upload for file /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set/train/dataset_info.json


...


Uploading: 100%|██████████| 125M/125M [00:21<00:00, 5.95MB/s] 

Upload successful: /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set/validation/data-00000-of-00001.arrow (119MB)
Starting upload for file /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set/validation/state.json



Uploading: 100%|██████████| 284/284 [00:00<00:00, 585B/s]

Upload successful: /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set/validation/state.json (284B)


HTTPError: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion